

# Compute cross-talk functions for LCMV beamformers

Visualise cross-talk functions at one vertex for LCMV beamformers computed
with different data covariance matrices, which affects their cross-talk
functions.


In [81]:
# Author: Olaf Hauk <olaf.hauk@mrc-cbu.cam.ac.uk>
#
# License: BSD-3-Clause
# Copyright the MNE-Python contributors.

In [82]:
import mne
from mne.beamformer import make_lcmv, make_lcmv_resolution_matrix
from mne.datasets import sample
from mne.minimum_norm import get_cross_talk, get_point_spread, resolution_metrics
import numpy as np


NOISE_COV = "adhoc" # "standard", "pre", "emptyroom", adhoc
COV_METHOD = "shrunk"  # "empirical", "shrunk"
UNIT_WEIGHT = True  
PLOT_RES = "psf" # ctf, psf, peak_err, sd_ext

print(__doc__)

data_path = sample.data_path()
subjects_dir = data_path / "subjects"
meg_path = data_path / "MEG" / "sample"
fname_fwd = meg_path / "sample_audvis-meg-eeg-oct-6-fwd.fif"
fname_cov = meg_path / "sample_audvis-cov.fif"
raw_empty_room_fname = meg_path / "ernoise_raw.fif"



fname_evo = meg_path / "sample_audvis-ave.fif"
raw_fname = meg_path / "sample_audvis_filt-0-40_raw.fif"

# Read raw data
raw = mne.io.read_raw_fif(raw_fname)

# only pick good EEG/MEG sensors
raw.info["bads"] += ["EEG 053"]  # bads + 1 more
picks = mne.pick_types(raw.info, meg=True, eeg=True, exclude="bads")

# Find events
events = mne.find_events(raw)

# event_id = {'aud/l': 1, 'aud/r': 2, 'vis/l': 3, 'vis/r': 4}
event_id = {"vis/l": 3, "vis/r": 4}

# raw.plot()



tmin, tmax = -0.2, 0.25  # epoch duration
epochs = mne.Epochs(
    raw,
    events,
    event_id=event_id,
    tmin=tmin,
    tmax=tmax,
    picks=picks,
    # baseline=(-0.2, 0.0),
    baseline=None,
    preload=True,
)

# covariance matrix for post-stimulus interval (around main evoked responses)
tmin, tmax = 0.05, 0.25
cov_post = mne.compute_covariance(epochs, tmin=tmin, tmax=tmax, method=COV_METHOD)
info = epochs.info
del epochs





if NOISE_COV == "emptyroom":
    print("Using empty room data for noise covariance estimation")
    raw_empty_room = mne.io.read_raw_fif(raw_empty_room_fname)
    # raw_empty_room.crop(0, 30)  # cropped just for speed
    raw_empty_room.info["bads"] = ["MEG 2443"]
    raw_empty_room.add_proj(raw.info["projs"])
    noise_cov = mne.compute_raw_covariance(raw_empty_room, method=COV_METHOD, rank="info")
    del raw_empty_room
    if COV_METHOD == "empirical":
        noise_cov = mne.cov.regularize(noise_cov, info, mag=0.1, grad=0.1, eeg=0.1, rank="info")

elif NOISE_COV == "standard":
    # read noise covariance matrix
    print('Reading noise covariance matrix from file: %s' % fname_cov)
    noise_cov = mne.read_cov(fname_cov)
    noise_cov = mne.cov.regularize(noise_cov, info, mag=0.1, grad=0.1, eeg=0.1, rank="info")

elif NOISE_COV == "pre":
    print("------------- Using pre-stimulus data for noise covariance estimation -------------")
    # covariance matrix for pre-stimulus interval
    tmin, tmax = -0.2, 0.0
    # noise_cov = mne.compute_covariance(epochs, tmin=tmin, tmax=tmax, method=COV_METHOD)
    noise_cov = mne.compute_raw_covariance(raw, tmin=215, method=COV_METHOD, rank="info", verbose=True)
    if COV_METHOD == "empirical":
        noise_cov = mne.cov.regularize(noise_cov, info, mag=0.1, grad=0.1, eeg=0.1, rank="info")

elif NOISE_COV == "adhoc":
    print("Using ad-hoc noise covariance matrix")
    noise_cov = mne.make_ad_hoc_cov(info)



del raw




Note: all executions are function-scoped as we do not assume the code below executes in an isolated kernel environment.

Opening raw data file /Users/harrisonritz/mne_data/MNE-sample-data/MEG/sample/sample_audvis_filt-0-40_raw.fif...
    Read a total of 4 projection items:
        PCA-v1 (1 x 102)  idle
        PCA-v2 (1 x 102)  idle
        PCA-v3 (1 x 102)  idle
        Average EEG reference (1 x 60)  idle
    Range : 6450 ... 48149 =     42.956 ...   320.665 secs
Ready.
Finding events on: STI 014
319 events found on stim channel STI 014
Event IDs: [ 1  2  3  4  5 32]
Not setting metadata
143 matching events found
No baseline correction applied
Created an SSP operator (subspace dimension = 4)
4 projection items activated
Loading data for 143 events and 69 original time points ...
0 bad epochs dropped
    Created an SSP operator (subspace dimension = 4)
    Setting small MEG eigenvalues to zero (without PCA)
    Setting small EEG eigenvalues to zero (without PCA)
Reducing data rank f

/var/folders/nl/tj__js1s2l7dkdwd16gh1xsc0000gn/T/ipykernel_97558/757560941.py:59: RuntimeWarning: Epochs are not baseline corrected, covariance matrix may be inaccurate
  cov_post = mne.compute_covariance(epochs, tmin=tmin, tmax=tmax, method=COV_METHOD)
/Users/harrisonritz/repos/mne-opm/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2374: RuntimeWarning: divide by zero encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/Users/harrisonritz/repos/mne-opm/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2374: RuntimeWarning: overflow encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/Users/harrisonritz/repos/mne-opm/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2374: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/Users/harrisonritz/repos/mne-opm/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2374: Runtim

Done.
Number of samples used : 4433
[done]
Using ad-hoc noise covariance matrix


/Users/harrisonritz/repos/mne-opm/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2374: RuntimeWarning: divide by zero encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/Users/harrisonritz/repos/mne-opm/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2374: RuntimeWarning: overflow encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/Users/harrisonritz/repos/mne-opm/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2374: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/Users/harrisonritz/repos/mne-opm/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2374: RuntimeWarning: divide by zero encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/Users/harrisonritz/repos/mne-opm/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2374: RuntimeWarning: overflow encountered in slog

## Compute LCMV filters with different data covariance matrices



In [83]:
# read forward solution
forward = mne.read_forward_solution(fname_fwd)



if UNIT_WEIGHT:
    # use forward operator with fixed source orientations
    mne.convert_forward_solution(forward, surf_ori=True, force_fixed=True, copy=False)

    # compute LCMV beamformer filters for pre-stimulus interval
    filters = make_lcmv(
        info,
        forward,
        cov_post,
        reg=0.05,
        noise_cov=noise_cov,
        pick_ori=None,
        rank=None,
        weight_norm=None,
        reduce_rank=False,
        verbose=False,
    )
else:
    # compute LCMV beamformer filters for pre-stimulus interval
    filters = make_lcmv(
        info,
        forward,
        cov_post,
        reg=0.05,
        noise_cov=noise_cov,
        pick_ori="max-power",
        rank="info",
        weight_norm="nai",
        reduce_rank=False,
        verbose=False,
        depth=None,
    )

    # use forward operator with fixed source orientations
    mne.convert_forward_solution(forward, surf_ori=True, force_fixed=True, copy=False)





Reading forward solution from /Users/harrisonritz/mne_data/MNE-sample-data/MEG/sample/sample_audvis-meg-eeg-oct-6-fwd.fif...
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    2 source spaces read
    Desired named matrix (kind = 3523 (FIFF_MNE_FORWARD_SOLUTION_GRAD)) not available
    Read MEG forward solution (7498 sources, 306 channels, free orientations)
    Desired named matrix (kind = 3523 (FIFF_MNE_FORWARD_SOLUTION_GRAD)) not available
    Read EEG forward solution (7498 sources, 60 channels, free orientations)
    Forward solutions combined: MEG, EEG
    Source spaces transformed to the forward solution coordinate frame
    Average patch normals will be employed in the rotation to the local surface coordinates....
    Converting to surface-based sou

In [84]:
# pick the ["precentral-lh"] from aparc label names

def _label_centroid_foci(label_name, parc, fs_subject, subjects_dir):
    """``(hemi, surface vertno)`` at a label's centroid, for a seed foci marker.

    Unlike :func:`_get_label_centroid_idx` this is not restricted to the
    source-space vertices, so it can mark the seed on any surface (e.g. the
    fsaverage group map) without needing a source space.  Returns ``None`` on
    failure.
    """
    try:
        labels = mne.read_labels_from_annot(
            fs_subject, parc=parc, subjects_dir=subjects_dir, verbose=False
        )
    except Exception as e:
        print(f"    WARNING: could not read {parc} labels for {fs_subject}: {e}")
        return None
    matched = [l for l in labels if l.name == label_name]
    if not matched:
        print(f"    WARNING: label '{label_name}' not found in {parc} for {fs_subject}")
        return None
    label = matched[0]
    centroid_vert = int(
        label.vertices[np.argmin(np.linalg.norm(label.pos - label.pos.mean(0), axis=1))]
    )
    return label.hemi, centroid_vert



group_foci = [
    _label_centroid_foci("precentral-lh", "aparc", "sample", subjects_dir),
    _label_centroid_foci("precentral-rh", "aparc", "sample", subjects_dir),
]

print(f"Group foci: {group_foci}")


Group foci: [('lh', 80015), ('rh', 77280)]


## Compute resolution matrices for the two LCMV beamformers



In [85]:
# compute cross-talk functions (CTFs) for one target vertex
# --- find source-space index for precentral-lh centroid ---
hemi_str, centroid_vert = group_foci[0]  # precentral-lh
hemi_idx = 0 if hemi_str == "lh" else 1
src = forward["src"][hemi_idx]

match = np.where(src["vertno"] == centroid_vert)[0]
if len(match):
    sources = [int(match[0])]
else:
    # centroid not in source space — find nearest by 3D position
    centroid_pos = src["rr"][centroid_vert]
    src_pos = src["rr"][src["vertno"]]
    sources = [int(np.argmin(np.linalg.norm(src_pos - centroid_pos, axis=1)))]

verttrue = [src["vertno"][sources[0]]]
print(f"Precentral-lh: src index {sources[0]}, vertex {verttrue[0]}")


resmat = make_lcmv_resolution_matrix(filters, forward, info)
if PLOT_RES == "ctf":
    stc = get_cross_talk(resmat, forward["src"], sources, norm=True)
elif PLOT_RES == "psf":
    stc = get_point_spread(resmat, forward["src"], sources, norm=True)
elif PLOT_RES == "peak_err":
    stc = resolution_metrics(resmat, forward["src"], "psf", metric="peak_err")
elif PLOT_RES == "sd_ext":
    stc = resolution_metrics(resmat, forward["src"], "psf", metric="sd_ext")
del resmat

print(f"sources: {sources}, verttrue: {verttrue}, stc_pre.data.shape: {stc.data.shape}")


Precentral-lh: src index 1982, vertex 81273
    364 out of 366 channels remain after picking
Dimensions of LCMV resolution matrix: (7498, 7498).
sources: [1982], verttrue: [np.int64(81273)], stc_pre.data.shape: (7498, 1)


## Visualize
Pre:



In [86]:
brain = stc.plot(
    "sample",
    "inflated",
    "split",
    views=["lateral", "medial"],
    subjects_dir=subjects_dir,
    figure=1,
    clim=dict(kind="value", lims=(0.10, 0.50, 1.0) if PLOT_RES in ("ctf", "psf") else (0, 5.0, 10.0)),
)

brain.add_text(
    0.1,
    0.9,
    f"cov={NOISE_COV} / method={COV_METHOD} / unit={UNIT_WEIGHT}",
    "title",
    font_size=14,
)

# mark true source location for CTFs
brain.add_foci(
    verttrue, coords_as_verts=True, scale_factor=1.0, hemi="lh", color="green"
)

Post:



The pre-stimulus beamformer's CTF has lower values in parietal regions
suppressed alpha activity?) but larger values in occipital regions (less
suppression of visual activity?).

